# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AyanButt1013/FlyRank_ML-Track_Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of Analysis & Time Window

* **Unit of Analysis (Grain):** One row = One unique content item (`content_hash_id`) aggregated at the page level.
* **Feature Window:** Daily performance metrics aggregated over a 90-day historical window (`2026-01-01` to `2026-03-31`).
* **Target/Outcome Window:** Observed traffic/visibility movement evaluated over the subsequent 30-day forward window (`2026-04-01` to `2026-04-30`) to avoid target leakage.
* **Snapshot Constraints:** Full daily facts stop at `2026-06-30` (3-day lag cutoff from the export date).

In [1]:
import duckdb
from google.colab import userdata

# Retrieve HF token safely
HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

# Verify grain and date windows in daily facts sample
query = """
SELECT
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date,
    COUNT(DISTINCT content_hash_id) AS total_unique_pages,
    COUNT(*) AS total_fact_rows
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet')
"""

df_window_check = con.sql(query).df()
print("--- WINDOW & GRAIN VERIFICATION ---")
print(df_window_check.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--- WINDOW & GRAIN VERIFICATION ---
  min_date   max_date  total_unique_pages  total_fact_rows
2026-06-01 2026-06-30              409205         11694072


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field Categorization Contract

* **Observed Features (Inputs):** `word_count`, `char_count`, `impressions_90d`, `clicks_90d`, `avg_position`, `ctr`, `content_age_days` (all computed strictly from the feature window).
* **Target / Proxy Label (Outputs):** `is_declining_label` defined as whether a page experienced a $>20\%$ drop in impressions/clicks in the forward 30-day window compared to the prior 90-day baseline.
* **Context Fields (Metadata & Grouping):** `content_hash_id`, `client_hash_id`, `content_type`, `category_count` (used for joins, grouping, and client-level holdout validation).
* **Excluded Fields (With Reasoning):**
  * `keyword_hash_id`, `url_hash_id`: Excluded from feature vector to prevent high-cardinality memorization and keep models generalized across clients.
  * Raw client names or domain identifiers: Omitted for privacy and data safety.
  * Product decision flags / scores (`health_score`, `priority_score`): Excluded to prevent circular reasoning and ensure predictions are built strictly from raw observable signals.

In [2]:
# Verify field presence and check null counts across feature candidates
query_fields = """
SELECT
    COUNT(content_hash_id) AS total_rows,
    COUNT(word_count) AS non_null_word_count,
    COUNT(content_updated_date) AS non_null_update_date,
    COUNT(content_type) AS non_null_type
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')
"""

df_fields = con.sql(query_fields).df()
print("--- FIELD COMPLETENESS AUDIT ---")
print(df_fields.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--- FIELD COMPLETENESS AUDIT ---
 total_rows  non_null_word_count  non_null_update_date  non_null_type
     519606               341838                519606         519606


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Empirical Query Verification

The code cell below verifies our contract claims by checking:
1. Primary key uniqueness for `dim_content` (`content_hash_id`).
2. Absence of duplicate rows at the content grain.
3. Distribution of missing values across key structural features.

In [3]:
# Audit primary key uniqueness and missing values
query_audit = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT content_hash_id) AS unique_content_ids,
    SUM(CASE WHEN word_count IS NULL THEN 1 ELSE 0 END) AS null_word_counts,
    SUM(CASE WHEN word_count = 0 THEN 1 ELSE 0 END) AS zero_word_counts
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')
"""

df_audit = con.sql(query_audit).df()
print("--- GRAIN & INTEGRITY AUDIT ---")
print(df_audit.to_string(index=False))

# Assert primary key contract holds
assert df_audit['total_rows'].values[0] == df_audit['unique_content_ids'].values[0], "Grain violation: content_hash_id is not unique!"
print("\n✅ Assertion Passed: One row = One content item in dim_content")

--- GRAIN & INTEGRITY AUDIT ---
 total_rows  unique_content_ids  null_word_counts  zero_word_counts
     519606              519606          177768.0               2.0

✅ Assertion Passed: One row = One content item in dim_content


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data Limitations & Boundary Conditions

1. **Unbalanced History:** Clients entered tracking at different dates (`gsc_data_start` vs. `ga4_data_start`), meaning earlier time frames contain search impressions but zero GA4 session/engagement data.
2. **GSC-Only Early Rows:** Rows where `ga4_data_available = FALSE` represent missing tracking infrastructure, not zero traffic.
3. **No Causal Interventions:** The dataset records passive search observations, not controlled experiments; we cannot infer that editing a page will guarantee performance recovery.
4. **Sparse Signals:** AI-referral sessions (`sessions_ai`) and scroll rate events are sparsely populated relative to standard search impressions.

In [4]:
# Query client history variance and GA4 tracking availability
query_limits = """
SELECT
    COUNT(DISTINCT client_hash_id) AS total_clients,
    MIN(gsc_data_start) AS earliest_gsc_start,
    MAX(gsc_data_start) AS latest_gsc_start,
    MIN(ga4_data_start) AS earliest_ga4_start
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')
"""

df_limits = con.sql(query_limits).df()
print("--- DATA LIMITS & HISTORY COVERAGE AUDIT ---")
print(df_limits.to_string(index=False))

--- DATA LIMITS & HISTORY COVERAGE AUDIT ---
 total_clients earliest_gsc_start latest_gsc_start earliest_ga4_start
           104         2025-01-27       2026-06-02         2025-10-29


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.